# 03 pretrain

Step 3 of the guide: pretrain a small GPT-2 decoder with next-token prediction (causal LM) on the step-2 token windows. Same architecture as the Wells Fargo 1B model, 100 times smaller.

| setting | Wells Fargo 1B | this run |
|---|---|---|
| layers | 16 | 6 |
| hidden size | 2,048 | 384 |
| attention heads | 32 | 6 |
| context length | 1,024 / 2,048 | 512 tokens (about 73 transactions) |
| positions | RoPE | GPT-2 learned absolute positions |
| objective | CLM main + MLM auxiliary | CLM only (MLM is an optional step-6 ablation) |
| checkpoint choice | lowest validation loss, never the last step | same (`load_best_model_at_end`, `metric_for_best_model=eval_loss`) |

Each stage below prints what it produced: the example the model actually sees, the parameter count, the prior each field has to beat, the checkpoints left on Drive, the loss curve and the per-field accuracy.

In [1]:
import os, sys, glob, shutil
from google.colab import drive, userdata
drive.mount("/content/drive")
ROOT = "/content/drive/MyDrive/Colab Notebooks/tfm"
tok = userdata.get("GH_TOKEN")
if not os.path.exists("/content/transaction-fm"):
    !git clone -q https://{tok}@github.com/lb603487485/transaction-fm.git /content/transaction-fm
else:
    !cd /content/transaction-fm && git pull -q
%pip install -q datasets xgboost
%cd /content/transaction-fm
sys.path.insert(0, "/content/transaction-fm/src")
os.makedirs(f"{ROOT}/notebooks", exist_ok=True)
for f in glob.glob("notebooks/*.ipynb"):          # new notebooks from the repo appear in Drive
    dst = f"{ROOT}/notebooks/{os.path.basename(f)}"
    if not os.path.exists(dst):
        shutil.copy(f, dst); print("new notebook in Drive:", os.path.basename(f))
import tfm
print("tfm ok")

Mounted at /content/drive
Mounted at /content/drive
/content/transaction-fm
/content/transaction-fm
tfm ok
tfm ok


## What trains on what, and how a Colab disconnect is survived

- The model sees `tokens_train.npy` only: windows cut from train rows, by step 2, never crossing a card or a split.
- `tokens_val.npy` is read for the validation loss every 500 steps; that loss picks the checkpoint. Test is not opened in this notebook.
- `output_dir` is on Drive and the Trainer saves every 500 steps. Stage 3c starts with `find_latest_checkpoint`: if the session died, rerun the notebook top to bottom and training continues from the newest complete checkpoint (model, optimizer, LR schedule, RNG, step counter and the loss history all come back). A checkpoint written half-way through a disconnect fails `verify_checkpoint` and is skipped.
- Keep one session under about 2 hours; the full run is about 8 epochs. Stage 3d prints every checkpoint left on Drive with its verification result.

## Stage 3a: the windows, and one example as the model sees it

In [2]:
import json, time, os, numpy as np, pandas as pd
pd.set_option("display.width", 200); pd.set_option("display.max_columns", 30)
from tfm.tokenizer import TransactionTokenizer
from tfm.pretrain import make_example, field_of, prior_accuracy, WindowDataset

P = f"{ROOT}/data/processed"
tok = TransactionTokenizer.load(f"{P}/tokenizer.json")
train_w, val_w = np.load(f"{P}/tokens_train.npy"), np.load(f"{P}/tokens_val.npy")
PAD, BOS = tok.vocab["<PAD>"], tok.vocab["<BOS>"]
fields = field_of(tok.id2tok)
SEQ = train_w.shape[1]
print(f"train windows {len(train_w):,} x {SEQ}    val windows {len(val_w):,}    vocab {len(tok.vocab)}    tokens per transaction {tok.tokens_per_tx}")
print(f"tokens seen per epoch {train_w.size:,}  (half-overlap windows, so about twice the raw train token count)")

meta = pd.read_parquet(f"{P}/windows_train_meta.parquet")
short = np.flatnonzero(meta["n_real"].to_numpy() < SEQ)
i = int(short[0]) if len(short) else 0                      # a padded window, so the pad handling is visible
n = int(meta["n_real"].iloc[i])
ex = make_example(train_w[i], PAD, BOS)
lab = lambda l: tok.decode([l])[0] if l != -100 else "-100"
def rows(sl):
    return pd.DataFrame({"pos": range(sl.start, sl.stop), "input_ids": tok.decode(ex["input_ids"][sl]),
                         "labels": [lab(l) for l in ex["labels"][sl]], "attention_mask": ex["attention_mask"][sl]})
print(f"\nexample: train window {i}, card {int(meta['card_id'].iloc[i])}, {n} real tokens of {SEQ}. Head (note <BOS> first) and the real-to-pad boundary:")
display(rows(slice(0, 9)))
display(rows(slice(max(n - 3, 0), min(n + 4, SEQ))))
print("the model predicts labels[t + 1] from input_ids[:t + 1]; -100 positions are left out of the loss")

train windows 31,856 x 512    val windows 1,407    vocab 567    tokens per transaction 7
tokens seen per epoch 16,310,272  (half-overlap windows, so about twice the raw train token count)
train windows 31,856 x 512    val windows 1,407    vocab 567    tokens per transaction 7
tokens seen per epoch 16,310,272  (half-overlap windows, so about twice the raw train token count)

example: train window 185, card 8, 56 real tokens of 512. Head (note <BOS> first) and the real-to-pad boundary:


,pos,input_ids,labels,attention_mask
0,0,<BOS>,<BOS>,1
1,1,<TX>,<TX>,1
2,2,DT_NA,DT_NA,1
3,3,HOUR_NIGHT,HOUR_NIGHT,1
4,4,CAT_shopping_net,CAT_shopping_net,1
5,5,"MER_fraud_Langworth, Boehm and Gulgowski","MER_fraud_Langworth, Boehm and Gulgowski",1
6,6,AMT_19,AMT_19,1
7,7,DIST_04,DIST_04,1
8,8,<TX>,<TX>,1



example: train window 185, card 8, 56 real tokens of 512. Head (note <BOS> first) and the real-to-pad boundary:


,pos,input_ids,labels,attention_mask
0,0,<BOS>,<BOS>,1
1,1,<TX>,<TX>,1
2,2,DT_NA,DT_NA,1
3,3,HOUR_NIGHT,HOUR_NIGHT,1
4,4,CAT_shopping_net,CAT_shopping_net,1
5,5,"MER_fraud_Langworth, Boehm and Gulgowski","MER_fraud_Langworth, Boehm and Gulgowski",1
6,6,AMT_19,AMT_19,1
7,7,DIST_04,DIST_04,1
8,8,<TX>,<TX>,1


,pos,input_ids,labels,attention_mask
0,53,CAT_shopping_net,CAT_shopping_net,1
1,54,"MER_fraud_Reichert, Rowe and Mraz","MER_fraud_Reichert, Rowe and Mraz",1
2,55,AMT_19,AMT_19,1
3,56,DIST_05,DIST_05,1
4,57,<PAD>,-100,0
5,58,<PAD>,-100,0
6,59,<PAD>,-100,0


the model predicts labels[t + 1] from input_ids[:t + 1]; -100 positions are left out of the loss


,pos,input_ids,labels,attention_mask
0,53,CAT_shopping_net,CAT_shopping_net,1
1,54,"MER_fraud_Reichert, Rowe and Mraz","MER_fraud_Reichert, Rowe and Mraz",1
2,55,AMT_19,AMT_19,1
3,56,DIST_05,DIST_05,1
4,57,<PAD>,-100,0
5,58,<PAD>,-100,0
6,59,<PAD>,-100,0


the model predicts labels[t + 1] from input_ids[:t + 1]; -100 positions are left out of the loss


## Stage 3b: the model, its size, and the prior each field has to beat

The prior is the accuracy of always guessing the most common train token of a field. A model that does not beat it has learned nothing about that field; the gap over the prior is what Wealthsimple reports instead of perplexity. The last line runs the whole test suite on this machine, including the tests that need `transformers` (GPT-2 config and the TrainingArguments warmup field, which changed name in transformers 5).

In [3]:
import torch
from transformers import GPT2LMHeadModel
from tfm.pretrain import gpt2_config

cfg = gpt2_config(len(tok.vocab), PAD, BOS)
model = GPT2LMHeadModel(cfg)
n_params = sum(p.numel() for p in model.parameters())
emb = (cfg.vocab_size + cfg.n_positions) * cfg.n_embd
print(f"GPT-2 decoder: {cfg.n_layer} layers, {cfg.n_embd} hidden, {cfg.n_head} heads, {cfg.n_positions} positions, vocab {cfg.vocab_size}")
print(f"parameters {n_params / 1e6:.2f}M  = embeddings {emb / 1e6:.2f}M + transformer blocks {(n_params - emb) / 1e6:.2f}M  (tied output layer)")
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE: Runtime -> Change runtime type -> T4")

prior = prior_accuracy(train_w, fields, PAD)
display(pd.Series(prior, name="prior accuracy").reindex(["all", "DT", "HOUR", "CAT", "MER", "AMT", "DIST", "special"]).to_frame().style.format("{:.3f}"))
!PYTHONPATH=src python -m pytest -q tests 2>&1 | tail -1

GPT-2 decoder: 6 layers, 384 hidden, 6 heads, 512 positions, vocab 567
parameters 11.06M  = embeddings 0.41M + transformer blocks 10.65M  (tied output layer)
GPU: Tesla T4


,prior accuracy
all,0.315
DT,0.220
HOUR,0.204
CAT,0.101
MER,0.168
AMT,0.148
DIST,0.362
special,1.000


33 passed in 12.40s


## Stage 3c: train (or resume)

`training_args` holds the guide's values: lr 5e-4, cosine schedule with 5 percent warmup, fp16, eval and save every 500 steps, keep 2 checkpoints, load the best by validation loss at the end. Batch 32 windows of 512 tokens. If this cell is interrupted, rerun the notebook from the top; the first print below says where it resumed.

Progress is printed as plain text: one line every 50 steps (step, loss, learning rate, elapsed, eta), one line per eval with the validation loss, perplexity and per-field accuracy, one line per checkpoint saved. Plain text survives a reconnect and shows on GitHub; the tqdm widget does neither.

In [ ]:
from pathlib import Path
from transformers import Trainer, PrinterCallback
from tfm.checkpointing import find_latest_checkpoint, verify_checkpoint, training_args
from tfm.pretrain import argmax_logits, compute_metrics_fn, print_progress_callback

OUT = f"{ROOT}/pretrain"
EPOCHS = 8
args = training_args(OUT, seed=0, num_train_epochs=EPOCHS, per_device_train_batch_size=32,
                     per_device_eval_batch_size=64, eval_accumulation_steps=8, dataloader_num_workers=2,
                     disable_tqdm=True)       # plain-text progress lines below instead of the widget
train_ds, val_ds = WindowDataset(train_w, PAD, BOS), WindowDataset(val_w, PAD, BOS)
trainer = Trainer(model=model, args=args, train_dataset=train_ds, eval_dataset=val_ds,
                  compute_metrics=compute_metrics_fn(fields), preprocess_logits_for_metrics=argmax_logits,
                  callbacks=[print_progress_callback()])
trainer.remove_callback(PrinterCallback)        # keep only the plain-text lines below

last = find_latest_checkpoint(OUT)
steps_per_epoch = len(train_ds) // args.per_device_train_batch_size
print(f"resuming from: {last.name if last else 'scratch'}    steps per epoch {steps_per_epoch:,}    total about {EPOCHS * steps_per_epoch:,}    eval+save every {args.save_steps}")
t0 = time.time()
trainer.train(resume_from_checkpoint=str(last) if last else None)
print(f"wall time this session {(time.time() - t0) / 60:.1f} min")

resuming from: checkpoint-4500    steps per epoch 995    total about 7,960    eval+save every 500
resuming from: checkpoint-4500    steps per epoch 995    total about 7,960    eval+save every 500


[transformers] There were missing keys in the checkpoint model loaded: ['lm_head.weight'].
[transformers] There were missing keys in the checkpoint model loaded: ['lm_head.weight'].


training: 7,968 steps total, starting at step 4,500, eval and save every 500 steps
training: 7,968 steps total, starting at step 4,500, eval and save every 500 steps


[transformers] `loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.
[transformers] `loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


step 4,550/7,968  57.1%  loss 1.639  lr 2.1e-04  elapsed 0.2 min  eta 17 min
step 4,550/7,968  57.1%  loss 1.639  lr 2.1e-04  elapsed 0.2 min  eta 17 min
{'loss': '1.639', 'grad_norm': '0.2007', 'learning_rate': '0.0002122', 'epoch': '4.568'}
{'loss': '1.639', 'grad_norm': '0.2007', 'learning_rate': '0.0002122', 'epoch': '4.568'}
step 4,600/7,968  57.7%  loss 1.639  lr 2.1e-04  elapsed 0.4 min  eta 15 min
step 4,600/7,968  57.7%  loss 1.639  lr 2.1e-04  elapsed 0.4 min  eta 15 min
{'loss': '1.639', 'grad_norm': '0.1596', 'learning_rate': '0.0002071', 'epoch': '4.618'}
{'loss': '1.639', 'grad_norm': '0.1596', 'learning_rate': '0.0002071', 'epoch': '4.618'}
step 4,650/7,968  58.4%  loss 1.640  lr 2.0e-04  elapsed 0.6 min  eta 14 min
step 4,650/7,968  58.4%  loss 1.640  lr 2.0e-04  elapsed 0.6 min  eta 14 min
{'loss': '1.64', 'grad_norm': '0.1772', 'learning_rate': '0.000202', 'epoch': '4.669'}
{'loss': '1.64', 'grad_norm': '0.1772', 'learning_rate': '0.000202', 'epoch': '4.669'}
step 4,7

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

... (18 outputs left out)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

{'eval_loss': '1.624', 'eval_acc_all': '0.4309', 'eval_acc_special': '1', 'eval_acc_DT': '0.2796', 'eval_acc_HOUR': '0.6944', 'eval_acc_CAT': '0.2216', 'eval_acc_MER': '0.1844', 'eval_acc_AMT': '0.2749', 'eval_acc_DIST': '0.3616', 'eval_runtime': '4.535', 'eval_samples_per_second': '310.3', 'eval_steps_per_second': '4.851', 'epoch': '7.53'}


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

saved checkpoint-7,500 to Drive
saved checkpoint-7,500 to Drive
step 7,550/7,968  94.8%  loss 1.631  lr 3.8e-06  elapsed 13.7 min  eta 2 min
step 7,550/7,968  94.8%  loss 1.631  lr 3.8e-06  elapsed 13.7 min  eta 2 min
{'loss': '1.631', 'grad_norm': '0.1422', 'learning_rate': '3.771e-06', 'epoch': '7.58'}
{'loss': '1.631', 'grad_norm': '0.1422', 'learning_rate': '3.771e-06', 'epoch': '7.58'}
step 7,600/7,968  95.4%  loss 1.630  lr 2.9e-06  elapsed 14.0 min  eta 2 min
step 7,600/7,968  95.4%  loss 1.630  lr 2.9e-06  elapsed 14.0 min  eta 2 min
{'loss': '1.63', 'grad_norm': '0.1418', 'learning_rate': '2.926e-06', 'epoch': '7.631'}
{'loss': '1.63', 'grad_norm': '0.1418', 'learning_rate': '2.926e-06', 'epoch': '7.631'}
step 7,650/7,968  96.0%  loss 1.634  lr 2.2e-06  elapsed 14.2 min  eta 1 min
step 7,650/7,968  96.0%  loss 1.634  lr 2.2e-06  elapsed 14.2 min  eta 1 min
{'loss': '1.634', 'grad_norm': '0.1354', 'learning_rate': '2.188e-06', 'epoch': '7.681'}
{'loss': '1.634', 'grad_norm': '0

## Stage 3d: what is left on Drive

Every checkpoint directory is verified (all files present, `global_step` matches the name). The best one by validation loss is saved as `pretrain/best` together with `tokenizer.json`: the vocabulary ships with the checkpoint, as the Wells Fargo deck insists, so step 4 cannot pair the weights with the wrong vocabulary.

In [ ]:
import shutil
for p in sorted(Path(OUT).glob("checkpoint-*"), key=lambda p: int(p.name.split("-")[1])):
    print(f"{p.name:18s}", verify_checkpoint(p) or "ok")
print("best checkpoint:", trainer.state.best_model_checkpoint, "   best eval_loss:", trainer.state.best_metric)
trainer.save_model(f"{OUT}/best")
shutil.copy(f"{P}/tokenizer.json", f"{OUT}/best/tokenizer.json")
print("pretrain/best:", sorted(os.listdir(f"{OUT}/best")))

## Stage 3e: loss curves and next-token accuracy per field

Left: train and validation loss by step, with the chosen checkpoint marked; if the dashed line is not at the far right the "best is not the last step" lesson showed up here too. Right: at the chosen checkpoint, how often the next token of each field is predicted exactly, next to the prior from Stage 3b. Perplexity is `exp(validation loss)`.

In [ ]:
import matplotlib.pyplot as plt
from tfm.pretrain import summarize

s = summarize(trainer.state.log_history, prior, n_params, len(train_ds), SEQ, EPOCHS)
tc, ec = np.array(s["train_curve"]), np.array(s["eval_curve"])
fig, ax = plt.subplots(1, 2, figsize=(13, 4))
ax[0].plot(tc[:, 0], tc[:, 1], label="train loss"); ax[0].plot(ec[:, 0], ec[:, 1], "o-", label="val loss")
ax[0].axvline(s["best_step"], ls="--", c="gray", label=f"best step {s['best_step']}")
ax[0].set_xlabel("step"); ax[0].set_ylabel("cross-entropy"); ax[0].set_title("loss"); ax[0].legend()
order = ["all", "DT", "HOUR", "CAT", "MER", "AMT", "DIST", "special"]
acc = pd.DataFrame({"model": s["per_field_accuracy_at_best"], "prior": s["prior_accuracy"]}).reindex(order).dropna()
acc.plot.bar(ax=ax[1], rot=0, title="next-token accuracy at the best step"); ax[1].set_ylim(0, 1)
plt.tight_layout(); fig.savefig(f"{ROOT}/results/03_pretrain.png", dpi=120); plt.show()

print(f"best step {s['best_step']}   val loss {s['best_eval_loss']:.4f}   perplexity {s['best_perplexity']:.2f}   last eval loss {s['last_eval_loss']:.4f}")
acc["gain over prior"] = acc["model"] - acc["prior"]
display(acc.style.format("{:.3f}"))

## Stage 3f: save the numbers

In [ ]:
s["wall_time_min_this_session"] = round((time.time() - t0) / 60, 1)
with open(f"{ROOT}/results/03_pretrain.json", "w") as f:
    json.dump(s, f, indent=2)
print(json.dumps({k: v for k, v in s.items() if k not in ("eval_curve", "train_curve")}, indent=2))

## What to read off the numbers

- `best step` vs the last step: if best is earlier, the WF point "checkpoints peak mid-run and then get worse" reproduced; say so in the report with both losses.
- `gain over prior` per field is the real result. Expect `special` and `HOUR` near 1 and near the prior (they are almost determined by position and by the previous tokens); `MER` and `AMT` are where a model earns its keep; `DT` says whether the model learned the card's rhythm.
- Perplexity is one number for the whole stream and is dominated by the easy fields; step 6 plots it against the fraud PR-AUC to test Wealthsimple's claim that the two correlate weakly.
- If val loss is still falling at the end, the run is short, not the model small: raise `EPOCHS` before drawing any conclusion ("short runs lie").

## Push results and this notebook back to GitHub

In [ ]:
import shutil, glob, os, json
os.makedirs("results/metrics", exist_ok=True); os.makedirs("results/figures", exist_ok=True)
for f in glob.glob(f"{ROOT}/results/*.json"):
    shutil.copy(f, "results/metrics/")
for f in glob.glob(f"{ROOT}/results/*.png"):
    shutil.copy(f, "results/figures/")
def same_code(a, b):   # a Drive notebook may be copied back only if its code cells equal the repo version
    cells = lambda p: ["".join(c["source"]) for c in json.load(open(p))["cells"] if c["cell_type"] == "code"]
    return cells(a) == cells(b)
for f in glob.glob(f"{ROOT}/notebooks/*.ipynb"):   # outputs are kept so GitHub shows them
    repo = f"notebooks/{os.path.basename(f)}"
    if not os.path.exists(repo) or same_code(f, repo):
        shutil.copy(f, repo)
    else:
        print("STALE, not copied:", os.path.basename(f), "-> delete it from Drive and rerun the setup cell")
!git config user.email "linboyy@hotmail.com" && git config user.name "Bo Lin"
!git add results notebooks && git commit -q -m "results and notebooks from Colab" && git push -q
!git log --oneline | head -1

In [ ]:
drive.flush_and_unmount()